# Topic modeling on persona_text (real seal package)
Unzip `seal_pkg.zip` (contains `seal/preprocessing`, `seal/prompting`, `seal/topic_modeling`) into this notebook's working directory, then run top to bottom.

In [ ]:
# --- Unpack the seal package (upload seal_pkg.zip to this Colab session first) ---
!unzip -o -q seal_pkg.zip

In [ ]:
# --- Dependencies ---
!pip install tomotopy plotly rbo gensim kneed python-dotenv ollama openai scikit-learn tqdm --quiet
!python -m spacy download en_core_web_lg --quiet

In [ ]:
import logging, os, json
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import sparse

from seal.topic_modeling import LDATopicModel
from seal.preprocessing import SimpleTMPreprocessor

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
logger = logging.getLogger("persona_tm")

## Load data and build `text` from `persona_text`

In [ ]:
df = pd.read_pickle("base_data_persona")

df["text"] = df["persona_text"].fillna("").astype(str)
df = df[df["text"] != ""].reset_index(drop=True)

if "id" not in df.columns:
    df["id"] = df.index.astype(str)

print(len(df), "documents")
df.head()

## Config
`stopword_files` is optional in this preprocessor - drop the arg entirely if you don't have custom stopword files; spaCy's own stopword list is already applied by default.

In [ ]:
MODELS_DIR = "models_persona"
NUM_ITERS = 1500
TOPN_WORDS = 15
MIN_DOC_WORDS = 2
TOP_N_REPRESENTATIVE = 10

LLM_PROVIDER = "openai"
LLM_MODEL = "gpt-4o-mini"
LLM_API_KEY = os.getenv("OPENAI_API_KEY")  # or set directly

os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
def build_preprocessor():
    return SimpleTMPreprocessor(
        spacy_model="en_core_web_lg",
        spacy_disable=["parser", "ner"],
        valid_pos=["NOUN", "VERB", "ADJ"],
        min_df=3,
        max_df=0.6,
        logger=logger,
        # stopword_files=["path/to/your_stops.txt"],  # optional
    )

## Build reference corpus (lemmatized, for coherence)

In [ ]:
df_lemmas = build_preprocessor().fit_transform(
    df, text_col="text", id_col="id", compute_bow=False, compute_tfidf=False,
)
reference_corpus = df_lemmas["lemmas"].tolist()

## Optimize number of topics
Trains one model per K in `TOPIC_RANGE`, scores each by coherence (c_NPMI), smooths the coherence-vs-K curve, and returns the peak(s). Every model is saved under `OPTIMIZE_DIR/model_{k}_topics/` regardless of whether it's selected, so nothing is wasted - and a `coherence_curve.html` plot is written to `OPTIMIZE_DIR` for a visual check.

This trains `len(TOPIC_RANGE)` full models, so it costs proportionally more time than a single fixed-K run - narrow the range once you have a rough sense of where coherence peaks.

In [ ]:
TOPIC_RANGE = range(5, 41, 5)  # adjust to your expected granularity
OPTIMIZE_DIR = Path(MODELS_DIR) / "optimize"

opt_results = LDATopicModel.optimize_num_topics(
    data=df[["id", "text"]].to_dict(orient="records"),
    base_path=OPTIMIZE_DIR,
    topic_range=TOPIC_RANGE,
    smoothing_window=3,
    n_best=3,
    logger=logger,
    num_iters=NUM_ITERS,
    topn=TOPN_WORDS,
    min_doc_words=MIN_DOC_WORDS,
    preprocessor=build_preprocessor(),
    reference_corpus=reference_corpus,
)

print("k -> mean coherence:")
for k, c in opt_results["scores"]:
    print(f"  {k:>3}: {c:.4f}")
print("\nSelected (coherence peaks):", opt_results["selected"])

## Pick K
Defaults to the top-coherence peak from `opt_results["selected"]`. Open `OPTIMIZE_DIR/coherence_curve.html` and override `CHOSEN_K` by hand if a different peak makes more sense qualitatively (e.g. topic counts that split cleanly vs. ones that don't).

In [ ]:
best = max(opt_results["selected"], key=lambda s: s["mean_coherence"])
CHOSEN_K = best["k"]
model_path = Path(best["model_path"])
print(f"Using K={CHOSEN_K}  (mean coherence={best['mean_coherence']:.4f})  ->  {model_path}")

## Load the chosen model
`optimize_num_topics` already trained and saved this model - no need to retrain.

In [ ]:
df_corpus = df[["id", "text"]]
lda = LDATopicModel.load(model_path, corpus=df_corpus)

coh = np.load(model_path / "TMmodel" / "topic_coherence.npy")
logger.info(f"  mean coherence (c_npmi) = {float(np.mean(coh)):.4f}")

## Generate topic labels via LLM
**Important:** set `llm_provider` / `llm_model_type` / `llm_api_key` on `lda.tm`, not on `lda`. `train()` builds `lda.tm` internally and it snapshots these values at that point - setting them on `lda` afterward has no effect.

In [ ]:
labels_file = model_path / "TMmodel" / "tpc_labels.txt"

if labels_file.exists():
    labels = labels_file.read_text(encoding="utf-8").splitlines()
    logger.info(f"Found existing tpc_labels.txt ({len(labels)} labels) - skipping labeling.")
else:
    logger.info(f"[{LLM_PROVIDER}] Generating topic labels via {LLM_MODEL} ...")
    lda.tm.llm_provider = LLM_PROVIDER
    lda.tm.llm_model_type = LLM_MODEL
    lda.tm.llm_api_key = LLM_API_KEY
    results = lda.tm.generate_topic_outputs(task="label", topn=TOPN_WORDS)
    labels = [lbl for _, lbl in sorted(results)]
    labels_file.write_text("\n".join(labels), encoding="utf-8")

for i, lbl in enumerate(labels):
    print(f"T{i}: {lbl}")

## Export document-topic assignments + representative examples + keywords

In [ ]:
thetas = lda.get_thetas()
df_out = df.copy()
df_out["dominant_topic"] = np.argmax(thetas, axis=1)
df_out["topic_label"] = df_out["dominant_topic"].map(lambda t: labels[t] if t < len(labels) else "")
df_out["topic_prob"] = thetas.max(axis=1)
df_out["topic_composition"] = [
    {i: round(float(p), 4) for i, p in enumerate(row)} for row in thetas
]

Kt = thetas.shape[1]
uniform_threshold = (1.0 / Kt) + 1e-4
df_out["low_confidence"] = df_out["topic_prob"] <= uniform_threshold
logger.info(
    f"  {int(df_out['low_confidence'].sum()):,} / {len(df_out):,} docs flagged "
    f"low_confidence (max theta <= {uniform_threshold:.4f})."
)

assignments_path = model_path / "document_topic_assignments.csv"
df_out[["id", "persona_text", "dominant_topic", "topic_label",
        "topic_prob", "low_confidence", "topic_composition"]].to_csv(assignments_path, index=False)
print("saved:", assignments_path)

In [ ]:
# representative persona_text examples per topic, ranked by theta
lda.tm.get_most_representative_per_tpc(
    sparse.csr_matrix(thetas), topn=TOP_N_REPRESENTATIVE, get_text=False,
)
text_by_id = dict(zip(df_out["id"], df_out["persona_text"]))

records = []
for t, docs in enumerate(lda.tm._most_representative_docs):
    for rank, (doc_id, _text, theta) in enumerate(docs, start=1):
        records.append({
            "topic_id": t,
            "topic_label": labels[t] if t < len(labels) else "",
            "rank": rank,
            "theta": round(float(theta), 4),
            "id": doc_id,
            "persona_text": text_by_id.get(doc_id, ""),
        })
repr_path = model_path / "topic_representative_examples.csv"
pd.DataFrame(records).to_csv(repr_path, index=False)
print("saved:", repr_path)

In [ ]:
topic_words = lda.get_topic_keys()
keywords_path = model_path / "topic_keywords.csv"
pd.DataFrame({
    "topic_id": range(len(labels)),
    "topic_label": labels,
    "n_docs": df_out["dominant_topic"].value_counts().reindex(range(len(labels)), fill_value=0).values,
    "top_words": [", ".join(w) for w in topic_words],
}).to_csv(keywords_path, index=False)
print("saved:", keywords_path)